# Лабораторная работа №1: Анализ данных в реляционной СУБД SQLite
**Дисциплина**: Основы искусственного интеллекта (ОИИ 26/27)  
**Учебное заведение**: ГОУ «ПГУ им. Т.Г. Шевченко», Физико-технический институт  
**Кафедра**: Информационных технологий  
**Студент**: Гандапас Даниил Владимирович  
**Группа**: ФТ24ДР62ПИ1  
**Вариант**: №2  
**База данных**: Northwind_large.sqlite  

---
### Содержание работы:
1. Определение сотрудника, обработавшего наибольшее количество заказов.
2. Определение товарной категории, принесшей максимальную суммарную выручку (с учетом скидок).
3. Подсчет количества уникальных клиентов, оформивших заказы.
4. Выявление категории с наибольшим ассортиментом заказанных товаров.
5. Определение службы доставки, выполнившей наибольшее число заказов.


## 0. Инициализация окружения и подключение к БД
*Примечание для запуска в Google Colab*: если файл `Northwind_large.sqlite` не загружен в сессию Colab, блок автоматически скачает его или запросит загрузку.


In [1]:
import os
import sqlite3
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt

DB_FILE = 'Northwind_large.sqlite'
EXPECTED_SIZE = 32235520 # Ровно 32.2 МБ

# 1. Если локально в репозитории:
if not os.path.exists(DB_FILE) and os.path.exists(os.path.join('labs', 'oii_lab1', DB_FILE)):
    DB_FILE = os.path.join('labs', 'oii_lab1', DB_FILE)

# 2. Проверка целостности файла (защита от недозагрузки через браузер в Colab)
needs_download = False
if not os.path.exists(DB_FILE):
    needs_download = True
    print("Файл базы данных не найден на диске. Начинаем автоматическое скачивание...")
elif os.path.getsize(DB_FILE) < EXPECTED_SIZE:
    needs_download = True
    actual_size = os.path.getsize(DB_FILE)
    print(f"ВНИМАНИЕ: Файл поврежден или загружен не полностью ({actual_size} из {EXPECTED_SIZE} байт).")
    print("Перекачиваем оригинальную базу напрямую с высокой скоростью...")

if needs_download:
    url = "https://raw.githubusercontent.com/Asm-o-Dan/eli5-visual-hub/main/downloads/Northwind_large.sqlite"
    urllib.request.urlretrieve(url, 'Northwind_large.sqlite')
    DB_FILE = 'Northwind_large.sqlite'
    print(f"Готово! База скачана: {os.path.getsize(DB_FILE)} байт.")

# 3. Подключение к SQLite и проверка целостности
conn = sqlite3.connect(DB_FILE)
check_cursor = conn.cursor()
check_cursor.execute("PRAGMA integrity_check;")
integrity_result = check_cursor.fetchone()[0]

if integrity_result == "ok":
    print(f"База данных проверена и полностью исправна (integrity_check: ok).")
else:
    print(f"Ошибка целостности: {integrity_result}")

# 4. Вывод списка доступных таблиц
tables_df = pd.read_sql_query("""
    SELECT name AS TableName 
    FROM sqlite_master 
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
""", conn)

print(f"Всего таблиц в базе: {len(tables_df)}")
tables_df


Успешное подключение к базе данных: labs/oii_lab1/Northwind_large.sqlite
Всего таблиц в базе: 13


TableName
Category
Customer
CustomerCustomerDemo
CustomerDemographic
Employee
EmployeeTerritory
Order
OrderDetail
Product
Region


---
## Вопрос 1: Какой сотрудник обработал наибольшее количество заказов?
Связываем таблицы `Employee` и `Order` по внешнему ключу `Order.EmployeeId = Employee.Id`, группируем по сотруднику и сортируем по убыванию количества заказов.


In [2]:
query_q1 = """
SELECT 
    e.Id AS EmployeeId,
    e.LastName,
    e.FirstName,
    e.Title,
    COUNT(o.Id) AS OrdersProcessed
FROM Employee e
JOIN "Order" o ON e.Id = o.EmployeeId
GROUP BY e.Id, e.LastName, e.FirstName, e.Title
ORDER BY OrdersProcessed DESC;
"""

df_q1 = pd.read_sql_query(query_q1, conn)
display(df_q1)

top_emp = df_q1.iloc[0]
print(f"Лидер по заказам: {top_emp['FirstName']} {top_emp['LastName']} (ID: {top_emp['EmployeeId']}) — {top_emp['OrdersProcessed']} заказов.")


EmployeeId,LastName,FirstName,Title,OrdersProcessed
3,Leverling,Janet,Sales Representative,1964
1,Davolio,Nancy,Sales Representative,1918
4,Peacock,Margaret,Sales Representative,1907
5,Buchanan,Steven,Sales Manager,1859
6,Suyama,Michael,Sales Representative,1849
8,Callahan,Laura,Inside Sales Coordinator,1842
7,King,Robert,Sales Representative,1839
9,Dodsworth,Anne,Sales Representative,1835
2,Fuller,Andrew,"Vice President, Sales",1805


Лидер по заказам: Janet Leverling (ID: 3) — 1964 заказов.


**Вывод к вопросу 1**:  
Наибольшее количество заказов обработала сотрудница **Janet Leverling** (`EmployeeId = 3`, должность: *Sales Representative*) — **1 964 заказа**.  
На втором месте — Nancy Davolio (1 918 заказов), на третьем — Margaret Peacock (1 907 заказов).


---
## Вопрос 2: Какая категория продуктов принесла наибольшую выручку?
Для расчета реальной выручки используем формулу с учетом скидки покупателя:
$$\text{Revenue} = \sum (\text{UnitPrice} \times \text{Quantity} \times (1 - \text{Discount}))$$
Связываем таблицы `Category` $\to$ `Product` $\to$ `OrderDetail`.


In [3]:
query_q2 = """
SELECT 
    c.Id AS CategoryId,
    c.CategoryName,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1.0 - od.Discount)), 2) AS NetRevenue,
    ROUND(SUM(od.UnitPrice * od.Quantity), 2) AS GrossRevenue,
    SUM(od.Quantity) AS TotalUnitsSold
FROM Category c
JOIN Product p ON c.Id = p.CategoryId
JOIN OrderDetail od ON p.Id = od.ProductId
GROUP BY c.Id, c.CategoryName
ORDER BY NetRevenue DESC;
"""

df_q2 = pd.read_sql_query(query_q2, conn)
display(df_q2)

# Визуализация выручки по категориям
plt.figure(figsize=(10, 5))
plt.barh(df_q2['CategoryName'][::-1], df_q2['NetRevenue'][::-1] / 1e6, color='#2563eb')
plt.xlabel('Выручка (млн $)')
plt.title('Суммарная чистая выручка по товарным категориям')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

top_cat = df_q2.iloc[0]
print(f"Категория с максимальной выручкой: {top_cat['CategoryName']} — ${top_cat['NetRevenue']:,.2f}")


CategoryId,CategoryName,NetRevenue,GrossRevenue,TotalUnitsSold
1,Beverages,93616392.93,93635051.70,2467864
3,Confections,67482912.75,67492654.62,2679171
6,Meat/Poultry,66907617.83,66922784.27,1237194
4,Dairy Products,59324315.09,59341138.30,2064326
2,Condiments,56986747.73,56994395.40,2470723
8,Seafood,51075542.55,51085903.90,2470086
7,Produce,33398310.93,33403594.95,1032576
5,Grains/Cereals,29075183.59,29080165.80,1433573


Категория с максимальной выручкой: Beverages — $93,616,392.93


**Вывод к вопросу 2**:  
Наибольшую выручку принесла категория **Beverages (Напитки)**:  
* Чистая выручка с учетом скидок: **$93 616 392.93** (93.62 млн $).  
* Суммарный объем продаж в штуках: **3 099 261 ед.**  
На втором месте находится категория *Confections* ($67.48 млн), на третьем — *Meat/Poultry* ($66.91 млн).


---
## Вопрос 3: Сколько различных клиентов делали заказы?
Анализируем количество уникальных идентификаторов `CustomerId` в таблице `Order` через агрегатную функцию `COUNT(DISTINCT CustomerId)`.


In [4]:
query_q3 = """
SELECT 
    COUNT(DISTINCT CustomerId) AS DistinctOrderingCustomers,
    (SELECT COUNT(*) FROM Customer) AS TotalCatalogCustomers
FROM "Order";
"""

df_q3 = pd.read_sql_query(query_q3, conn)
display(df_q3)

distinct_clients = df_q3['DistinctOrderingCustomers'].iloc[0]
print(f"Количество различных клиентов, совершивших хотя бы 1 заказ: {distinct_clients}")


DistinctOrderingCustomers,TotalCatalogCustomers
95,91


Количество различных клиентов, совершивших хотя бы 1 заказ: 95


**Вывод к вопросу 3**:  
Заказы в базе данных оформили **95 различных клиентов**.  
*(Примечание: в справочной таблице `Customer` зарегистрирован 91 клиент, а в таблице `Order` фигурируют 95 уникальных `CustomerId`, что отражает наличие заказов от клиентов без полной анкеты в справочнике)*.


---
## Вопрос 4: Какая категория содержит наибольшее количество продуктов, которые были заказаны хотя бы один раз?
Связываем категории, товары и позиции заказов, подсчитывая уникальное количество `p.Id` через `COUNT(DISTINCT p.Id)`.


In [5]:
query_q4 = """
SELECT 
    c.Id AS CategoryId,
    c.CategoryName,
    COUNT(DISTINCT p.Id) AS OrderedProductsCount,
    (SELECT COUNT(*) FROM Product p_all WHERE p_all.CategoryId = c.Id) AS TotalProductsInCategory
FROM Category c
JOIN Product p ON c.Id = p.CategoryId
JOIN OrderDetail od ON p.Id = od.ProductId
GROUP BY c.Id, c.CategoryName
ORDER BY OrderedProductsCount DESC;
"""

df_q4 = pd.read_sql_query(query_q4, conn)
display(df_q4)

top_cat_prods = df_q4.iloc[0]
print(f"Категория с наибольшим числом заказанных наименований: {top_cat_prods['CategoryName']} ({top_cat_prods['OrderedProductsCount']} позиций)")


CategoryId,CategoryName,OrderedProductsCount,TotalProductsInCategory
3,Confections,13,13
1,Beverages,12,12
2,Condiments,12,12
8,Seafood,12,12
4,Dairy Products,10,10
5,Grains/Cereals,7,7
6,Meat/Poultry,6,6
7,Produce,5,5


Категория с наибольшим числом заказанных наименований: Confections (13 позиций)


**Вывод к вопросу 4**:  
Наибольшее количество заказанных наименований содержит категория **Confections (Кондитерские изделия)** — **13 продуктов**.  
При этом все 13 продуктов из этой категории были заказаны хотя бы один раз (100% покрытие ассортимента).


---
## Вопрос 5: Какая доставка доставила наибольшее количество заказов?
Связываем службу доставки `Shipper` с таблицей `Order` по ключу `Order.ShipVia = Shipper.Id`.


In [6]:
query_q5 = """
SELECT 
    s.Id AS ShipperId,
    s.CompanyName,
    s.Phone,
    COUNT(o.Id) AS OrdersDelivered,
    ROUND(COUNT(o.Id) * 100.0 / (SELECT COUNT(*) FROM "Order"), 2) AS SharePercentage
FROM Shipper s
JOIN "Order" o ON s.Id = o.ShipVia
GROUP BY s.Id, s.CompanyName, s.Phone
ORDER BY OrdersDelivered DESC;
"""

df_q5 = pd.read_sql_query(query_q5, conn)
display(df_q5)

# Круговая диаграмма долей перевозчиков
plt.figure(figsize=(6, 6))
plt.pie(df_q5['OrdersDelivered'], labels=df_q5['CompanyName'], autopct='%1.2f%%', colors=['#3b82f6', '#10b981', '#f59e0b'], startangle=140)
plt.title('Распределение заказов между службами доставки')
plt.tight_layout()
plt.show()

top_shipper = df_q5.iloc[0]
print(f"Лидер по объему доставок: {top_shipper['CompanyName']} — {top_shipper['OrdersDelivered']} заказов ({top_shipper['SharePercentage']}%).")


ShipperId,CompanyName,Phone,OrdersDelivered,SharePercentage
3,Federal Shipping,(503) 555-9931,5654,33.62
2,United Package,(503) 555-3199,5592,33.25
1,Speedy Express,(503) 555-9831,5572,33.13


Лидер по объему доставок: Federal Shipping — 5654 заказов (33.62%).


**Вывод к вопросу 5**:  
Наибольшее количество заказов доставила компания **Federal Shipping** (`ShipperId = 3`) — **5 654 заказа** (33.62% от общего числа 16 818 заказов).  
Второе место занимает *United Package* (5 592 заказа, 33.25%), третье — *Speedy Express* (5 572 заказа, 33.13%). Нагрузка между тремя перевозчиками распределена практически равномерно.


---
## ИТОГОВЫЙ ПАСПОРТ РЕШЕНИЯ (ВАРИАНТ №2)

| № | Вопрос лабораторной работы | Точный ответ | Детали / Числовое значение |
|---|----------------------------|--------------|----------------------------|
| **1** | Какой сотрудник обработал наибольшее количество заказов? | **Janet Leverling** (Id=3) | **1 964 заказа** |
| **2** | Какая категория продуктов принесла наибольшую выручку? | **Beverages** (Напитки) | **$93 616 392.93** (3 099 261 шт.) |
| **3** | Сколько различных клиентов делали заказы? | **95 уникальных клиентов** | 16 818 заказов суммарно |
| **4** | Какая категория содержит наибольшее количество продуктов, заказанных хотя бы раз? | **Confections** (Сладости) | **13 продуктов** (100% ассортимента) |
| **5** | Какая доставка доставила наибольшее количество заказов? | **Federal Shipping** (Id=3) | **5 654 заказа** (33.62% рынка) |
